In [1]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(ggpubr)
#     library(BSgenome)
    library(biomaRt)
#     library(BSgenome.Ocuniculus.NCBI.oryCun2)
#     library(GenomicFeatures)
#     library(compEpiTools)
#     library(GenomeInfoDb)
#     library(stringi)
#     library(AnnotationHub)
    library(Matrix)
    library(SingleCellExperiment)
    library(Seurat)
    library(reshape2)
    library(scater)
    library(viridis)
})

options(repr.plot.width=15, repr.plot.height=8)

In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC'
io$output.directory <- file.path(io$basedir,"ArchR")
io$plot.dir = file.path(io$output.directory,'Plots')
setwd(io$output.directory)

### Load ArchR object

In [3]:
io$archR.directory = file.path(io$output.directory, 'Project/')

ArchRProject.filt = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

### Prepare scRNA-seq data

In [4]:
# rna_in = '/rds/project/rds-SDzz0CATGms/users/bt392/0X_Rabbit_ATAC/RNA/'
# counts_in = file.path(rna_in, 'counts.mtx')
# genes_in = file.path(rna_in, 'genes.csv')
# meta_in1 = file.path(rna_in, 'meta.csv')
# meta_in2 = file.path('/rds/project/rds-SDzz0CATGms/users/bt392/0X_Rabbit_ATAC/metadata.csv')
# sizefactors_in = file.path(rna_in, 'normalisation/sizefactors.tab')

In [5]:
rna_in = '/rds/project/rds-SDzz0CATGms/users/mlnt2/PhD_MT06/'
counts_in = file.path(rna_in, '3_cellqc/raw_counts.mtx')
genes_in = file.path(rna_in, '2_cellcalling/genes.tsv')
meta_in1 = file.path(rna_in, '5_doublet/meta.tab')
meta_in2 = file.path('/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC/RNA/metadata.csv')
sizefactors_in = file.path(rna_in, 'sizefactors.tab')


In [6]:
counts = readMM(counts_in)

In [7]:
genes = fread(genes_in, header=FALSE)
meta = fread(meta_in1)

In [8]:
colnames(counts) = meta$cell
rownames(counts) = genes$V1

In [9]:
# Keep only cells that are not doublet or stripped nuclei
keep = meta[meta$doublet==FALSE & meta$stripped==FALSE,]$cell
counts = counts[,keep]
meta = meta[meta$cell %in% keep,]

In [10]:
# load new metadata
meta = fread(meta_in2)[,-1]
rownames(meta) = meta$cell

In [11]:
sce = SingleCellExperiment(assays = list("counts" = counts))

In [12]:
sfs = meta$sizeFactor
sizeFactors(sce) = sfs
sce = scater::logNormCounts(sce)

In [13]:
summary(meta$cell == colnames(counts))

   Mode    TRUE 
logical  146133 

In [14]:
seRNA = CreateSeuratObject(counts = counts(sce), # counts(sce) used before, testing to see how this affects anything
                         min.cells = 20,
                         meta.data = meta)

Warning message:
“The following arguments are not used: row.names”


In [ ]:
seRNA[['logcounts']] = CreateAssayObject(counts = logcounts(sce))
DefaultAssay(seRNA) = 'logcounts'

In [ ]:
seRNA

In [ ]:
rna_in = '/rds/project/rds-SDzz0CATGms/users/bt392/0X_Rabbit_ATAC/RNA/'
saveRDS(seRNA, file.path(rna_in, 'seurat.rds'))

### Integrate scATAC with scRNA

### Mapping by stage

In [ ]:
rna_in = '/rds/project/rds-SDzz0CATGms/users/mlnt2/PhD_MT06/'
seRNA = readRDS(file.path(rna_in, 'seurat.rds'))

In [ ]:
ArchRProject.filt@geneAnnotation$genes$symbol = ArchRProject.filt@geneAnnotation$genes$gene_id

In [ ]:
stages = data.frame('sample' = c('rabbit_BGRGP1', 'rabbit_BGRGP2', 'rabbit_BGRGP3', 'rabbit_BGRGP4', 'rabbit_BGRGP5', 'rabbit_BGRGP6', 'rabbit_BGRGP7', 'rabbit_BGRGP8'),
                    'stage_mapping' = c('GD7', 'GD8', 'GD8', 'GD9', 'GD9', 'GD9', 'GD9', 'GD9' ))
cells = rownames(ArchRProject.filt@cellColData)
ArchRProject.filt@cellColData = merge(ArchRProject.filt@cellColData, stages, by='sample')
rownames(ArchRProject.filt@cellColData) = cells

In [ ]:
mapping = function(stage_select){
    # Subset RNA
    RNA_stage = subset(x = seRNA, subset = stage == stage_select)
    message(stage_select)
    # Subset ATAC 
    keep = rownames(ArchRProject.filt@cellColData[ArchRProject.filt@cellColData$stage_mapping==stage_select,])
    ATAC_stage = ArchRProject.filt[keep]
    ATAC_stage = suppressWarnings(addGeneScoreMatrix(ATAC_stage, force=TRUE))
    
    # Integrate
    ATAC_stage <- addGeneIntegrationMatrix(
        ArchRProj = ATAC_stage, 
        useMatrix = "GeneScoreMatrix",
        matrixName = "GeneIntegrationMatrix",
        reducedDims = "IterativeLSI_Harmony",
        seRNA = RNA_stage,
        addToArrow = TRUE,
        force= TRUE,
        groupRNA = "celltype",
        nameCell = "predictedCell_Un",
        nameGroup = "predictedGroup_celltype",
        nameScore = "predictedScore_celltype")
    
    # Extract mapping
    meta = as.data.frame(ATAC_stage@cellColData[, c('predictedCell_Un', 'predictedGroup_celltype', 'predictedScore_celltype')])
    meta$cell = rownames(meta)
    return(meta)
}

In [ ]:
mapped = suppressWarnings(lapply(unique(seRNA@meta.data$stage), mapping))
mapped = rbindlist(mapped)

In [ ]:
write.csv(mapped, file.path(io$output.directory, 'celltype_mapping.csv'), row.names=FALSE)

In [ ]:
mapped = mapped[match(mapped$cell, rownames(ArchRProject.filt@cellColData)),]
rownames(mapped) = mapped$cell

In [ ]:
umap = getEmbedding(ArchRProject.filt, 'UMAP')
colnames(umap) = c('umap1', 'umap2')

In [ ]:
test = merge(umap, mapped, by.x=0, by.y='cell')

In [ ]:
head(test)

In [ ]:
ggplot(test, aes(umap1, umap2, col=predictedGroup_celltype)) + 
    geom_point() + 
    theme_bw() + 
    theme(legend.position='none')

In [ ]:
mapped = as.data.frame(mapped)
rownames(mapped) = mapped$cell
mapped$cell=NULL

In [ ]:
tail(mapped)

In [ ]:
summary(rownames(ArchRProject.filt@cellColData) == rownames(mapped))

In [ ]:
ArchRProject.filt@cellColData = cbind(ArchRProject.filt@cellColData, mapped)

In [ ]:
p1 <- plotEmbedding(
    ArchRProject.filt, 
    colorBy = "cellColData", 
    name = "predictedGroup_celltype"
) + theme_void()

p2 <- plotEmbedding(
    ArchRProject.filt, 
    colorBy = "cellColData", 
    name = "predictedScore_celltype"
) + theme_void()

p3<- plotEmbedding(
    ArchRProject.filt, 
    colorBy = "cellColData", 
    name = "stage"
) + theme_void()

p4<- plotEmbedding(
    ArchRProject.filt, 
    colorBy = "cellColData", 
    name = "Clusters"
) + theme_void()

In [ ]:
plotPDF(
  p1 + theme_void(),
  name = "celltype",
    height = 10, width = 10
)

plotPDF(
  p2 + theme_void(),
  name = "celltype_score"
)

plotPDF(
  p3 + theme_void(),
  name = "stage"
)

plotPDF(
  p4 + theme_void(),
  name = "Clusters"
)

In [ ]:
meta_RNA = fread(meta_in2)[,-1]

In [ ]:
pdf1 = table(meta_RNA$celltype, meta_RNA$stage)
pdf1 = as.data.frame(sweep(pdf1, 2, colSums(pdf1), "/"))
pdf1$Var3 = 'RNA'

pdf2 = table(ArchRProject.filt@cellColData$predictedGroup_celltype, ArchRProject.filt@cellColData$stage)
pdf2 = as.data.frame(sweep(pdf2, 2, colSums(pdf2), "/"))
pdf2$Var3 = 'ATAC'

props = rbind(pdf1, pdf2)
props = props[,c('Var1', 'Var2', 'Var3', 'Freq')]
props$Freq = as.numeric(props$Freq)

In [ ]:
options(repr.plot.width=15, repr.plot.height=15)
p = ggplot(props, aes(Var1, Freq, fill=Var3)) + 
    geom_bar(stat='identity', position = "dodge") + 
    facet_wrap(~Var2, ncol=1) + 
    theme_bw() + 
    ylab('Proportion') + 
    theme(axis.text.x=element_text(angle=-90, vjust=0, hjust=0),
          axis.title.x=element_blank(),
         text = element_text(size=20)) 
p

In [ ]:
pdf(sprintf("%s/celltype_proportions.pdf",io$plot.dir), width=14, height=12)
print(p)
dev.off()

In [ ]:
pdf2 = table(ArchRProject.filt@cellColData$predictedGroup_celltype, ArchRProject.filt@cellColData$Clusters)
pdf2 = as.data.frame(sweep(pdf2, 2, colSums(pdf2), "/"))

In [ ]:
test = dcast(pdf2, Var1 ~ Var2, value.var='Freq')
rownames(test) = test$Var1
test$Var1 = NULL
test <- scale(t(test))

In [ ]:
ord <- hclust( dist(test, method = "euclidean"), method = "ward.D" )$order

In [ ]:
pdf2$Var2 <- factor(pdf2$Var2, levels = rownames(test)[ord])


In [ ]:
options(repr.plot.width=15, repr.plot.height=10)

p = ggplot(pdf2, aes(Var2, Var1, fill=Freq)) + 
    geom_tile() + 
    scale_fill_viridis() + 
    theme(text=element_text(size = 15),
         axis.title = element_blank(),
         legend.position='none') 
p 

pdf(sprintf("%s/celltype_per_cluster.pdf",io$plot.dir), width=14, height=12)
print(p)
dev.off()

In [ ]:
reannotation = data.frame(
    Clusters = paste0('C', 1:22),
    celltype_manual = c('Erythroid', #1
                       'Venous_endothelium',#2
                       'Venous_endothelium',#3
                       'Erythroid',#4
                       'Lateral_plate_mesoderm',#5
                       'Lateral_plate_mesoderm_mesenchyme',#6
                       'Presomitic_mesoderm',#7
                       'Epiblast',#8
                       'Unkown',#9
                       'Epiblast',#10
                       'Hindbrain_spinal_cord',#11
                       'Epiblast',#12
                       'Pharyngeal_endoderm',#13
                       'YS_endothelium',#14
                       'Hypoblast',
                       'Hypoblast',#16
                       'Hypoblast',#17
                       'Hypoblast',
                       'Syncytiotrophoblast',
                       'Syncytiotrophoblast progenitors',
                       'Syncytiotrophoblast progenitors',
                       'Cytotrophoblast')) #22

In [ ]:
ArchRProject.filt@cellColData$cell = rownames(ArchRProject.filt@cellColData)

In [ ]:
ArchRProject.filt@cellColData = merge(ArchRProject.filt@cellColData, reannotation, by='Clusters')

In [ ]:
head(cells)

In [ ]:
ArchRProject.filt@cellColData = ArchRProject.filt@cellColData[order(ArchRProject.filt@cellColData$cell, cells),]
rownames(ArchRProject.filt@cellColData) = ArchRProject.filt@cellColData$cell

In [ ]:
rownames(ArchRProject.filt@cellColData) = cells

In [ ]:
p5 <- plotEmbedding(
    ArchRProject.filt, 
    colorBy = "cellColData", 
    name = "celltype_manual"
) + theme_void()

plotPDF(
  p5 + theme_void(),
  name = "celltype_manual"
)

In [ ]:
markerGenes = c('ENSOCUG00000017835')

p1 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneIntegrationMatrix", 
    name = markerGenes, 
    continuousSet = "horizonExtra",
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
) 

p2 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneScoreMatrix", 
    continuousSet = "horizonExtra",
    name = markerGenes, 
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
)

In [ ]:
cowplot::plot_grid(p1,p2)

### Fix pseudo expression data

In [ ]:
mapped_cells = ArchRProject.filt@cellColData$predictedCell_Un
seRNA_mapped = seRNA[,mapped_cells]
#counts = seRNA_mapped@assays$logcounts@counts
counts = seRNA_mapped@assays$RNA@counts
colnames(counts) = rownames(ArchRProject.filt@cellColData)

In [ ]:
counts = counts[rownames(counts) %in% ArchRProject.filt@geneAnnotation$genes$gene_id, ]

In [ ]:
counts = as.matrix(counts)

In [ ]:
genes = rownames(counts) 

In [ ]:
rowranges = ArchRProject.filt@geneAnnotation$genes
rowranges = rowranges[rowranges$gene_id %in% genes,]

In [ ]:
counts = counts[order(match(rownames(counts), rowranges$gene_id)),]

In [ ]:
sceRNA <- SummarizedExperiment(assays=SimpleList(counts=counts),rowRanges=rowranges[,0])

In [ ]:
ArchRProject.filt = addGeneExpressionMatrix(
  input = ArchRProject.filt,
  seRNA = sceRNA)

In [ ]:
getAvailableMatrices(ArchRProj = ArchRProject.filt)

In [ ]:
ArchRProject.filt = addImputeWeights(
  ArchRProj = ArchRProject.filt,
  reducedDims = "IterativeLSI_Harmony")

In [ ]:
markerGenes = c('ENSOCUG00000017835')

p1 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneExpressionMatrix", 
    name = markerGenes, 
    continuousSet = "horizonExtra",
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
) 

p2 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneScoreMatrix", 
    continuousSet = "horizonExtra",
    name = markerGenes, 
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
)

In [ ]:
cowplot::plot_grid(p1,p2)

In [ ]:
ArchRProject.filt <- saveArchRProject(ArchRProj = ArchRProject.filt)

In [ ]:
ArchRProject.filt = addGeneExpressionMatrix(
  input = ArchRProject.filt,
  seRNA = sceRNA)

In [ ]:
markerGenes = c('ENSOCUG00000025597')

p1 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneIntegrationMatrix", 
    name = markerGenes, 
    continuousSet = "horizonExtra",
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
) 

p2 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneScoreMatrix", 
    continuousSet = "horizonExtra",
    name = markerGenes, 
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
)

In [ ]:
mapped = mapped[[1]]

In [ ]:
head(mapped)

In [ ]:
head(colnames(seRNA))

In [ ]:
mapped_cells = mapped$predictedCell_Un
seRNA_mapped = seRNA[,mapped_cells]
colnames(seRNA_mapped) = rownames(mapped)

In [ ]:
length(colnames(seRNA_mapped))

In [ ]:
length(rownames(mapped))

In [ ]:
seRNA_mapped

In [ ]:
sceRNA = Seurat::as.SingleCellExperiment(seRNA_mapped)

In [ ]:
seRNA = seRNA[rownames(seRNA) %in% ArchRProject.filt@geneAnnotation$genes$gene_id ,]

In [ ]:
length(unique(ArchRProject.filt@geneAnnotation$genes$gene_id))

In [ ]:
ArchRProject.filt@geneAnnotation$genes$symbol = ArchRProject.filt@geneAnnotation$genes$gene_id

In [ ]:
ArchRProject.filt@geneAnnotation$genes

In [ ]:
ArchRProject.filt = addGeneScoreMatrix(ArchRProject.filt, force=TRUE)

In [ ]:
test = getFeatures(ArchRProject.filt, 'TileMatrix')

In [ ]:
test = getMatrixFromProject(
  ArchRProj = ArchRProject.filt,
  useMatrix = "GeneScoreMatrix")

In [ ]:
ArchRProject.filt <- addGeneIntegrationMatrix(
    ArchRProj = ArchRProject.filt, 
    useMatrix = "GeneScoreMatrix",
 #   useMatrix = "TileMatrix",
    matrixName = "GeneIntegrationMatrix",
    reducedDims = "IterativeLSI_Harmony",
    seRNA = seRNA,
    addToArrow = TRUE,
    force= TRUE,
    groupRNA = "celltype",
    nameCell = "predictedCell_Un",
    nameGroup = "predictedGroup_celltype",
    nameScore = "predictedScore_celltype"
)

In [ ]:
cM <- as.matrix(confusionMatrix(ArchRProject.filt$Clusters, ArchRProject.filt$predictedGroup_lltype))
preClust <- colnames(cM)[apply(cM, 1 , which.max)]
cbind(preClust, rownames(cM))

In [ ]:
umap = getEmbedding(ArchRProj = ArchRProject.filt, embedding = "UMAP")
colnames(umap) = c('umap1', 'umap2')

In [ ]:
metadata = ArchRProject.filt@cellColData

In [ ]:
plot = as.data.frame(merge(umap, metadata, by=0))

In [ ]:
p1 = ggplot(plot, aes(umap1, umap2, col=predictedGroup_celltype)) +
    geom_point() + 
    theme_void()
    

In [ ]:
p1

In [ ]:
plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "cellColData", 
    name = 'predictedGroup_celltype', 
    embedding = "UMAP"
) + theme_void()

In [ ]:
getAvailableMatrices(ArchRProject.filt)

In [ ]:
ArchRProject.filt <- addImputeWeights(ArchRProject.filt)

In [ ]:
ArchRProject.filt@geneAnnotation$genes

In [ ]:
head(gene_map[gene_map$external_gene_name=='NKX2-5',])

In [ ]:
markerGenes  <- c(
#     "ENSOCUG00000032657",
#     'ENSOCUG00000021510',
#     'ENSOCUG00000009672',
#     'ENSOCUG00000033269',
#     'ENSOCUG00000039498',
#     'ENSOCUG00000021075',
#     'ENSOCUG00000031211',
#     'ENSOCUG00000015176',
#     'ENSOCUG00000036263'
    'ENSOCUG00000017835', # DNMT3B > epiblast
    'ENSOCUG00000025597', # GATA1 -> blood
    'ENSOCUG00000009246', # GCM1 --> SCT 
    'ENSOCUG00000026343', # VTCN1 --> amniotic ectoderm
    'ENSOCUG00000000041', # NKX2-5 > heart
    'ENSOCUG00000010829', # MYL4 > cardiomyocytes
    'ENSOCUG00000015170', # HOXA11 > Allantois
    'ENSOCUG00000017248', # TAL1
    'ENSOCUG00000000451', # EOMES
    'ENSOCUG00000002633', # OTX2 > neural
    'ENSOCUG00000004752', # SIX6 > neural
    'ENSOCUG00000014268', # Six1 > neural
    'ENSOCUG00000005360', # Noto > Notochord
    'ENSOCUG00000007084', # CHR > notochord
    'ENSOCUG00000011487', # DKK1 > Gut/AVE
    'ENSOCUG00000010192' # CDH5 > endothelium
    
  )

# p1 <- plotEmbedding(
#     ArchRProj = ArchRProject.filt, 
#     colorBy = "GeneIntegrationMatrix", 
#     name = markerGenes, 
#     continuousSet = "horizonExtra",
#     embedding = "UMAP",
#     imputeWeights = getImputeWeights(ArchRProject.filt)
# ) 

p2 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneScoreMatrix", 
    continuousSet = "horizonExtra",
    name = markerGenes, 
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
)

In [ ]:
do.call(cowplot::plot_grid, c(list(ncol = 2), p2))

In [ ]:
p2c <- lapply(p2, function(x){
    x +  theme(plot.margin = unit(c(0, 0, 0, 0), "cm")) + theme_void() + theme(text = element_text(size=20), legend.position='none')
})

options(repr.plot.width=15, repr.plot.height=10)
do.call(cowplot::plot_grid, c(list(ncol = 1), p2c))

In [ ]:
options(repr.plot.width=15, repr.plot.height=10)
cowplot::plot_grid(p2c[[15]], p2c[[16]])

In [ ]:
cowplot::plot_grid(p2, ncol=2)

In [ ]:
meta_RNA = fread(meta_in2)[,-1]

In [ ]:
prop_RNA = as.data.frame(table(meta_RNA$celltype))
prop_RNA$Var2 = 'RNA'

prop_ATAC = as.data.frame(table(ArchRProject.filt@cellColData$predictedGroup_celltype))
prop_ATAC$Var2 = 'ATAC'

props = rbind(prop_RNA, prop_ATAC)
props = props[,c('Var1', 'Var2', 'Freq')]
props$Freq = as.numeric(props$Freq)

In [ ]:
props[props$Var2 == 'RNA',]$Freq = props[props$Var2 == 'RNA',]$Freq / nrow(meta_RNA)
props[props$Var2 == 'ATAC',]$Freq = props[props$Var2 == 'ATAC',]$Freq / nrow(ArchRProject.filt@cellColData)


In [ ]:
ggplot(props, aes(Var1, Freq, fill=Var2, group.by=Var2)) + 
    geom_bar(stat='identity', position = "dodge") + 
   # scale_x_discrete(breaks = unique(props$Var1)[order(unique(meta$Var1))]) +
    theme_bw() + 
    theme(axis.text.x=element_text(angle=-90, vjust=0, hjust=0)) 

In [ ]:
do.call(cowplot::plot_grid, c(list(ncol = 3), p2c))

In [ ]:
ArchRProject.filt@geneAnnotation$genes$symbol = ArchRProject.filt@geneAnnotation$genes$gene_id

In [ ]:
colnames(ArchRProject.filt@cellColData)

In [ ]:
p = plotBrowserTrack(
    ArchRProj = ArchRProject.filt, 
    groupBy = "celltype_manual", 
    geneSymbol = 'ENSOCUG00000025597', 
    upstream = 20000,
    downstream = 20000,
    useMatrix = 'GeneExpressionMatrix'
)

In [ ]:
grid::grid.newpage()
grid::grid.draw(p[[1]])

In [ ]:
ArchRProject.filt@geneAnnotation$exons

# Integration with ranged summarized experiment instead of seurat object

In [4]:
rna_in = '/rds/project/rds-SDzz0CATGms/users/mlnt2/PhD_MT06/'
seRNA = readRDS(file.path(rna_in, 'seurat.rds'))

In [5]:
test = seRNA[,1:4000]

In [6]:
test

An object of class Seurat 
51339 features across 4000 samples within 2 assays 
Active assay: logcounts (30725 features, 0 variable features)
 1 other assay present: RNA

In [7]:
seRNA_mapped = test
counts = seRNA_mapped@assays$RNA@counts

In [8]:
counts = counts[rownames(counts) %in% ArchRProject.filt@geneAnnotation$genes$gene_id, ]

In [9]:
genes = rownames(counts) 

In [10]:
rowranges = ArchRProject.filt@geneAnnotation$genes
rowranges = rowranges[rowranges$gene_id %in% genes,]

In [11]:
rowranges = rowranges[order(match(rowranges$gene_id, rownames(counts))),]

In [12]:
sceRNA <- SummarizedExperiment(assays=SimpleList(counts=counts),rowRanges=rowranges[,0], colData=seRNA_mapped@meta.data)

In [13]:
test = ArchRProject.filt[1:4000,]

Dropping ImputeWeights Since You Are Subsetting Cells! ImputeWeights is a cell-x-cell Matrix!



In [ ]:
test <- addGeneIntegrationMatrix(
    ArchRProj = test, 
    useMatrix = "GeneScoreMatrix",
 #   useMatrix = "TileMatrix",
    matrixName = "GeneIntegrationMatrix",
    reducedDims = "IterativeLSI_Harmony",
    seRNA = sceRNA,
    addToArrow = TRUE,
    force= TRUE,
    groupRNA = "celltype",
    nameCell = "predictedCell_Un",
    nameGroup = "predictedGroup_celltype",
    nameScore = "predictedScore_celltype"
)

ArchR logging to : ArchRLogs/ArchR-addGeneIntegrationMatrix-115b8c696286b-Date-2021-12-04_Time-17-56-55.log
If there is an issue, please report to github with logFile!

2021-12-04 17:56:55 : Running Seurat's Integration Stuart* et al 2019, 0.011 mins elapsed.

2021-12-04 17:56:56 : Checking ATAC Input, 0.03 mins elapsed.

2021-12-04 17:56:56 : Checking RNA Input, 0.03 mins elapsed.

2021-12-04 17:56:58 : Found 14712 overlapping gene names from gene scores and rna matrix!, 0.058 mins elapsed.

2021-12-04 17:56:58 : Creating Integration Blocks, 0.058 mins elapsed.

2021-12-04 17:56:58 : Prepping Interation Data, 0.058 mins elapsed.

2021-12-04 17:56:58 : Computing Integration in 1 Integration Blocks!, 0 mins elapsed.

2021-12-04 17:56:58 : Block (1 of 1) : Computing Integration, 0 mins elapsed.

2021-12-04 17:56:59 : Block (1 of 1) : Identifying Variable Genes, 0.004 mins elapsed.

2021-12-04 17:57:00 : Block (1 of 1) : Getting GeneScoreMatrix, 0.028 mins elapsed.

2021-12-04 17:57:08 : 

In [ ]:
markerGenes = c('ENSOCUG00000017835')

p1 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneIntegrationMatrix", 
    name = markerGenes, 
    continuousSet = "horizonExtra",
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
) 

p2 <- plotEmbedding(
    ArchRProj = ArchRProject.filt, 
    colorBy = "GeneScoreMatrix", 
    continuousSet = "horizonExtra",
    name = markerGenes, 
    embedding = "UMAP",
    imputeWeights = getImputeWeights(ArchRProject.filt)
)

In [ ]:
cowplot::plot_grid(p1,p2)